# Pre-Consultation Agent - Kaggle GPU Deployment

This notebook deploys your pre-consultation agent on Kaggle's **FREE Tesla P100 GPU** (better than Colab's T4!).

## Setup Checklist:
1. ✅ Turn on GPU: **Settings → Accelerator → GPU P100**
2. ✅ Enable Internet: **Settings → Internet → ON**
3. ✅ Add Secrets (optional but recommended):
   - Go to **Add-ons → Secrets**
   - Add `GEMINI_API_KEY` and `HF_TOKEN`

**GPU Quota**: 30 hours/week (more generous than Colab!)

Let's get started! 🚀

## Step 1: Verify GPU and Environment

In [ ]:
import torch
import os

print("🔍 Checking environment...\n")

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f}GB")
else:
    print("❌ No GPU detected!")
    print("   Go to Settings → Accelerator → GPU P100")

# Check Python version
import sys
print(f"\n🐍 Python: {sys.version.split()[0]}")

# Check working directory
print(f"📁 Working dir: {os.getcwd()}")

## Step 2: Clone Repository from GitHub

In [ ]:
import os
from pathlib import Path

# Remove if already exists
if Path('/kaggle/working/pre-consultation-agent').exists():
    !rm -rf /kaggle/working/pre-consultation-agent

# Clone from GitHub
print("📥 Cloning repository...\n")
!git clone https://github.com/buwituze/pre-consultation-agent.git /kaggle/working/pre-consultation-agent

# Navigate to backend
os.chdir('/kaggle/working/pre-consultation-agent/backend')
print(f"\n✅ Repository cloned!")
print(f"📁 Current directory: {os.getcwd()}")

In [ ]:
# Pull latest changes from GitHub (if repository already exists)
import os
from pathlib import Path

backend_path = Path('/kaggle/working/pre-consultation-agent/backend')

if backend_path.exists():
    os.chdir('/kaggle/working/pre-consultation-agent')
    print("🔄 Pulling latest changes from GitHub...\n")
    !git pull origin main
    
    print("\n✅ Latest code pulled!")
    print("\n📌 Current commit:")
    !git log -1 --oneline
    print("\n📅 Commit details:")
    !git log -1 --pretty=format:"Commit: %h%nAuthor: %an%nDate: %ar%nMessage: %s"
    
    # Navigate back to backend
    os.chdir(str(backend_path))
    print(f"\n📁 Working directory: {os.getcwd()}")
else:
    print("⚠️ Repository not found. Run the clone cell above first.")

## Step 3: Install Dependencies

This installs all required packages including transformers, torch, librosa, fastapi, etc.

In [ ]:
# Install requirements
print("📦 Installing dependencies...\n")
!pip install -q -r requirements.txt

print("\n✅ Dependencies installed!")

# Verify key packages
import transformers
import librosa
import fastapi
print(f"\n📚 Key versions:")
print(f"   transformers: {transformers.__version__}")
print(f"   librosa: {librosa.__version__}")
print(f"   fastapi: {fastapi.__version__}")

## Step 4: Configure Environment Variables

**⚠️ IMPORTANT**: Add your API keys below OR use Kaggle Secrets (recommended).

### Using Kaggle Secrets (Recommended):
1. Click **Add-ons → Secrets** in the right sidebar
2. Add these secrets:
   - `GEMINI_API_KEY`: Your Google Gemini API key
   - `HF_TOKEN`: Your Hugging Face token
3. Enable them for this notebook

### Manual Setup:
If you don't use Secrets, replace the values below.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Try to get from Kaggle Secrets first
try:
    user_secrets = UserSecretsClient()
    gemini_key = user_secrets.get_secret("GEMINI_API_KEY")
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("✅ Using Kaggle Secrets")
except:
    print("⚠️ Kaggle Secrets not found - using manual setup")
    # Manual setup (replace these!)
    gemini_key = 'YOUR_GEMINI_API_KEY_HERE'
    hf_token = 'YOUR_HF_TOKEN_HERE'

# Set environment variables
os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['HF_TOKEN'] = hf_token
os.environ['DEVICE'] = 'cuda'  # Use GPU!
os.environ['MAX_TURNS'] = '6'

# Verify
print("\n✅ Environment configured:")
print(f"   GEMINI_API_KEY: {'✓ SET' if gemini_key and 'YOUR_' not in gemini_key else '❌ NOT SET'}")
print(f"   HF_TOKEN: {'✓ SET' if hf_token and 'YOUR_' not in hf_token else '❌ NOT SET'}")
print(f"   DEVICE: {os.environ['DEVICE']}")

if 'YOUR_' in gemini_key or 'YOUR_' in hf_token:
    print("\n⚠️ WARNING: Please update API keys before continuing!")

## Step 5: Load Whisper Models

This downloads and loads both Whisper models (~6GB total). **First time takes 3-5 minutes**.

The models will be cached for future runs.

In [ ]:
import sys
import time
sys.path.insert(0, '/kaggle/working/pre-consultation-agent/backend')

from models import model_a

print("🔄 Loading Whisper models on GPU...")
print("   This takes 3-5 minutes on first run (downloading ~6GB)\n")

start = time.time()
model_a.load_models()
status = model_a.get_models_status()
elapsed = time.time() - start

print(f"\n✅ Models loaded in {elapsed:.1f}s")
print(f"   Status: {status}")

# Check GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"\n📊 GPU Memory:")
    print(f"   Allocated: {allocated:.2f}GB")
    print(f"   Reserved: {reserved:.2f}GB")

## Step 6: Test Transcription with Audio File

**How to upload your audio file:**
1. Click **"+ Add Data"** button (right sidebar)
2. Click **"Upload"** → **"New Dataset"**
3. Upload your WAV file (e.g., `Benitha_testfile.wav`)
4. Name your dataset (e.g., "test-audio")
5. The file will be available at: `/kaggle/input/test-audio/Benitha_testfile.wav`

**Expected Performance**: ~5 seconds for 24-second audio on P100 GPU

In [ ]:
import time
import os
import glob

print("📁 Looking for WAV files in uploaded datasets...\n")

# Automatically find WAV files in /kaggle/input/
wav_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.wav'):
            wav_files.append(os.path.join(root, file))

if wav_files:
    print(f"✅ Found {len(wav_files)} WAV file(s):")
    for i, filepath in enumerate(wav_files, 1):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"   {i}. {os.path.basename(filepath)} ({size_mb:.2f}MB)")
        print(f"      Path: {filepath}")
    
    # Use the first WAV file found
    audio_file_path = wav_files[0]
    print(f"\n🎤 Using: {os.path.basename(audio_file_path)}\n")
    
    # Read audio file
    with open(audio_file_path, 'rb') as f:
        audio_bytes = f.read()
    
    print(f"   Size: {len(audio_bytes)/(1024*1024):.2f}MB")
    
    # Run transcription
    start = time.time()
    result = model_a.transcribe(audio_bytes, language_hint='kinyarwanda')
    elapsed = time.time() - start
    
    print(f"\n✅ Transcription completed in {elapsed:.1f}s")
    print(f"   Language: {result['dominant_language']}")
    print(f"   Confidence: {result['mean_confidence']:.2%}")
    print(f"\n📝 Transcription:")
    print(f"   {result['full_text']}")
    
    # GPU memory
    if torch.cuda.is_available():
        print(f"\n📊 GPU Memory Used: {torch.cuda.memory_allocated(0)/1e9:.2f}GB")
else:
    print("❌ No WAV files found in uploaded datasets!")
    print("\n📤 To upload your audio file:")
    print("   1. Click '+ Add Data' button (right sidebar)")
    print("   2. Click 'Upload' → 'New Dataset'")
    print("   3. Upload your WAV file (e.g., Benitha_testfile.wav)")
    print("   4. Name your dataset (e.g., 'test-audio')")
    print("   5. Re-run this cell")
    print("\n💡 Your file will be at: /kaggle/input/<dataset-name>/Benitha_testfile.wav")

## Step 7: Test Multiple Audio Files

Test all your uploaded audio files. If you have multiple WAV files uploaded, they'll all be transcribed.

In [ ]:
import time
import os

print("🎤 Testing all uploaded WAV files...\n")

# Find all WAV files
wav_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.wav'):
            wav_files.append(os.path.join(root, file))

if wav_files:
    print(f"✅ Found {len(wav_files)} audio file(s)\n")
    print("="*70)
    
    for i, audio_path in enumerate(wav_files, 1):
        print(f"\n🎵 File {i}/{len(wav_files)}: {os.path.basename(audio_path)}")
        print(f"   Path: {audio_path}")
        
        # Get file info
        file_size = os.path.getsize(audio_path) / (1024 * 1024)
        print(f"   Size: {file_size:.2f} MB")
        
        # Read audio
        with open(audio_path, 'rb') as f:
            audio_bytes = f.read()
        
        # Transcribe
        print(f"\n   🔄 Transcribing...")
        start = time.time()
        result = model_a.transcribe(audio_bytes, language_hint='kinyarwanda')
        elapsed = time.time() - start
        
        # Results
        print(f"   ✅ Complete in {elapsed:.2f}s")
        print(f"\n   📊 Results:")
        print(f"      Language: {result['dominant_language']}")
        print(f"      Confidence: {result['mean_confidence']:.2%}")
        print(f"\n   📝 Transcription:")
        print(f"      {result['full_text']}")
        
        # GPU usage
        if torch.cuda.is_available():
            gpu_mem = torch.cuda.memory_allocated(0) / 1e9
            print(f"\n   💾 GPU Memory: {gpu_mem:.2f}GB")
        
        print("\n" + "="*70)
else:
    print("❌ No WAV files found!")
    print("\n📤 Upload your audio file:")
    print("   1. Click '+ Add Data' (right sidebar)")
    print("   2. Click 'Upload' → 'New Dataset'  ")
    print("   3. Upload your WAV file")
    print("   4. Re-run this cell")

## Step 8: Test with Different Languages

Test the same audio with different language hints to see how it affects transcription.

In [ ]:
import time

# Find the first WAV file
wav_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.wav'):
            wav_files.append(os.path.join(root, file))

if wav_files:
    audio_path = wav_files[0]
    print(f"🎤 Testing language detection on: {os.path.basename(audio_path)}\n")
    
    # Read audio once
    with open(audio_path, 'rb') as f:
        audio_bytes = f.read()
    
    # Test with different language hints
    languages = ['kinyarwanda', 'english', None]
    
    print("="*70)
    for lang in languages:
        lang_display = lang if lang else "auto-detect"
        print(f"\n🌍 Testing with language hint: {lang_display}")
        
        start = time.time()
        result = model_a.transcribe(audio_bytes, language_hint=lang)
        elapsed = time.time() - start
        
        print(f"   ⏱️  Time: {elapsed:.2f}s")
        print(f"   🔍 Detected: {result['dominant_language']}")
        print(f"   📊 Confidence: {result['mean_confidence']:.2%}")
        print(f"   📝 Text: {result['full_text']}")
        print()
    
    print("="*70)
else:
    print("❌ No audio files found. Upload a WAV file first.")

## Step 9: Performance Benchmark

Run multiple transcriptions to measure average performance and GPU efficiency.

In [ ]:
import time
import statistics

# Find first WAV file
wav_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.wav'):
            wav_files.append(os.path.join(root, file))

if wav_files:
    audio_path = wav_files[0]
    file_size = os.path.getsize(audio_path) / (1024 * 1024)
    
    print(f"🏃 Performance Benchmark")
    print(f"File: {os.path.basename(audio_path)} ({file_size:.2f} MB)\n")
    
    # Read audio
    with open(audio_path, 'rb') as f:
        audio_bytes = f.read()
    
    # Run multiple times
    num_runs = 3
    times = []
    
    print(f"Running {num_runs} iterations...\n")
    for i in range(num_runs):
        print(f"Run {i+1}/{num_runs}...", end=" ")
        start = time.time()
        result = model_a.transcribe(audio_bytes, language_hint='kinyarwanda')
        elapsed = time.time() - start
        times.append(elapsed)
        print(f"✅ {elapsed:.2f}s")
    
    # Statistics
    avg_time = statistics.mean(times)
    min_time = min(times)
    max_time = max(times)
    
    print(f"\n📊 Performance Summary:")
    print(f"   Average time: {avg_time:.2f}s")
    print(f"   Min time: {min_time:.2f}s")
    print(f"   Max time: {max_time:.2f}s")
    
    # GPU stats
    if torch.cuda.is_available():
        gpu_mem = torch.cuda.memory_allocated(0) / 1e9
        gpu_max = torch.cuda.max_memory_allocated(0) / 1e9
        print(f"\n💾 GPU Memory:")
        print(f"   Current: {gpu_mem:.2f}GB")
        print(f"   Peak: {gpu_max:.2f}GB")
    
    print(f"\n🎯 Result Quality:")
    print(f"   Language: {result['dominant_language']}")
    print(f"   Confidence: {result['mean_confidence']:.2%}")
    print(f"   Text length: {len(result['full_text'])} chars")
else:
    print("❌ No audio files found. Upload a WAV file first.")

## Summary

**What we've done:**
1. ✅ Verified GPU access (Tesla P100)
2. ✅ Cloned your repository from GitHub
3. ✅ Installed all dependencies
4. ✅ Loaded Whisper models on GPU (~6GB)
5. ✅ Tested audio transcription with Model A directly
6. ✅ Benchmarked performance across multiple runs

**Performance on Tesla P100:**
- **GPU**: Tesla P100 (16GB VRAM)
- **Expected Speed**: ~5 seconds for 24-second audio
- **Weekly Quota**: 30 hours/week

**What This Tests:**
- ✅ Audio transcription (Model A only)
- ✅ Language detection (Kinyarwanda/English)
- ✅ GPU acceleration
- ✅ Confidence scoring

**What This Doesn't Test:**
- ❌ Database functionality (no PostgreSQL on Kaggle)
- ❌ FastAPI server (not needed for audio testing)
- ❌ Full conversation pipeline (Models B-F)

**For Full System Testing:**
- Use local development with database setup
- See `SETUP.md` for complete backend setup
- Use `API_DOCS.md` for API reference

**Next Steps:**
- Upload your WAV files and test transcription
- Compare performance with different audio lengths
- Test both Kinyarwanda and English audio

Enjoy your FREE GPU transcription! 🎉